# Segment Anything — from scratch

SAM reimplemented from the paper, loading Meta's official ViT-B checkpoint with **zero key mismatches**.

Runtime → Change runtime type → **T4 GPU** before running.

## Setup

In [ ]:
!git clone -qhttps://github.com/Kshitiz-2002/sam-from-scratch.git
%cd sam-from-scratch
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
print('ready')

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

from model.image_encoder import ImageEncoderViT
from model.prompt_encoder import PromptEncoder
from model.transformer import TwoWayTransformer
from model.mask_decoder import MaskDecoder
from model.sam import SAM

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')

## Build the model

ViT-B config from the paper: 12 blocks, 14×14 windowed attention, global attention at blocks 2/5/8/11.

In [ ]:
sam = SAM(
    image_encoder=ImageEncoderViT(
        img_size=1024, patch_size=16, in_chans=3,
        embed_dim=768, depth=12, num_heads=12, mlp_ratio=4.0,
        out_chans=256, qkv_bias=True,
        use_abs_pos=True, use_rel_pos=True,
        window_size=14, global_attn_indexes=(2, 5, 8, 11),
    ),
    prompt_encoder=PromptEncoder(
        embed_dim=256,
        image_embedding_size=(64, 64),
        input_image_size=(1024, 1024),
        mask_in_chans=16,
    ),
    mask_decoder=MaskDecoder(
        transformer_dim=256,
        transformer=TwoWayTransformer(
            depth=2, embedding_dim=256, num_heads=8, mlp_dim=2048
        ),
    ),
)

n = sum(p.numel() for p in sam.parameters())
print(f'{n/1e6:.1f}M parameters')

## Load Meta's checkpoint

`strict=True` here — every tensor in the 375MB official checkpoint maps onto this implementation.

In [ ]:
state = torch.load('sam_vit_b_01ec64.pth', map_location='cpu')
sam.load_state_dict(state)          # no strict=False, no remapping
sam = sam.to(DEVICE).eval()
print('loaded')

## Helpers

In [ ]:
def load_image(path, long_side=1024):
    """Resize so the longest side is 1024, keeping aspect ratio."""
    img = np.array(Image.open(path).convert('RGB'))
    h, w = img.shape[:2]
    scale = long_side / max(h, w)
    nh, nw = int(h * scale + 0.5), int(w * scale + 0.5)

    t = torch.as_tensor(img).permute(2, 0, 1)[None].float().to(DEVICE)
    t = F.interpolate(t, (nh, nw), mode='bilinear', align_corners=False)[0]
    return img, t, (h, w), scale


def show(img, masks, ious, points=None, box=None):
    n = len(masks)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))
    axes = np.atleast_1d(axes)
    for i, ax in enumerate(axes):
        ax.imshow(img)
        m = np.zeros((*masks[i].shape, 4))
        m[masks[i]] = [1.0, 0.36, 0.15, 0.55]
        ax.imshow(m)
        if points is not None:
            for (x, y), lab in points:
                ax.scatter([x], [y], c='lime' if lab else 'red',
                           s=180, marker='*', edgecolors='white', linewidths=1)
        if box is not None:
            import matplotlib.patches as pt
            x0, y0, x1, y1 = box
            ax.add_patch(pt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                      fill=False, edgecolor='lime', lw=2))
        ax.set_title(f'IoU {ious[i]:.3f} · {masks[i].sum():,} px', fontsize=11)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## Load an image

Replace with your own, or upload via the file browser on the left.

In [ ]:
IMAGE = 'assets/dog.jpg'

img, img_t, orig_size, scale = load_image(IMAGE)
plt.figure(figsize=(8, 6)); plt.imshow(img); plt.axis('off'); plt.show()
print('original size', orig_size)

## Point prompt

One click is ambiguous — it could mean the object, a part of it, or the whole group containing it. SAM returns three candidates and ranks them with its IoU head.

In [ ]:
CLICK = (200, 400)      # (x, y) in ORIGINAL image coordinates

out = sam([{
    'image': img_t,
    'original_size': orig_size,
    'point_coords': torch.tensor([[[CLICK[0] * scale, CLICK[1] * scale]]],
                                 dtype=torch.float, device=DEVICE),
    'point_labels': torch.tensor([[1]], device=DEVICE),
}], multimask_output=True)[0]

masks = out['masks'][0].cpu().numpy()
ious = out['iou_predictions'][0].cpu().numpy()
show(img, masks, ious, points=[(CLICK, 1)])

## Multiple points

Green stars include, red exclude. With more than one prompt the ambiguity mostly disappears, so a single mask is returned.

In [ ]:
PTS = [((200, 400), 1), ((150, 300), 1)]     # ((x, y), label) — 1 include, 0 exclude

coords = torch.tensor([[[x * scale, y * scale] for (x, y), _ in PTS]],
                      dtype=torch.float, device=DEVICE)
labels = torch.tensor([[l for _, l in PTS]], device=DEVICE)

out = sam([{
    'image': img_t,
    'original_size': orig_size,
    'point_coords': coords,
    'point_labels': labels,
}], multimask_output=False)[0]

show(img,
     out['masks'][0].cpu().numpy(),
     out['iou_predictions'][0].cpu().numpy(),
     points=PTS)

## Box prompt

Boxes are encoded as two corner tokens with their own learned type embeddings, separate from the point ones.

In [ ]:
BOX = (30, 130, 330, 534)      # x0, y0, x1, y1 in ORIGINAL coordinates

out = sam([{
    'image': img_t,
    'original_size': orig_size,
    'boxes': torch.tensor([BOX], dtype=torch.float, device=DEVICE) * scale,
}], multimask_output=False)[0]

show(img,
     out['masks'][0].cpu().numpy(),
     out['iou_predictions'][0].cpu().numpy(),
     box=BOX)

## Encode once, prompt many times

The image encoder is ~90% of the compute and runs once per image. Every additional prompt reuses that cached embedding — this is what makes SAM interactive.

In [ ]:
import time

with torch.no_grad():
    batched = sam.preprocess(img_t)[None]

    torch.cuda.synchronize()
    t0 = time.time()
    embedding = sam.image_encoder(batched)
    torch.cuda.synchronize()
    t_encode = time.time() - t0

    coords = torch.tensor([[[200 * scale, 400 * scale]]], dtype=torch.float, device=DEVICE)
    labels = torch.tensor([[1]], device=DEVICE)

    torch.cuda.synchronize()
    t0 = time.time()
    sparse, dense = sam.prompt_encoder(points=(coords, labels), boxes=None, masks=None)
    low_res, iou = sam.mask_decoder(
        image_embeddings=embedding,
        image_pe=sam.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse,
        dense_prompt_embeddings=dense,
        multimask_output=True,
    )
    torch.cuda.synchronize()
    t_prompt = time.time() - t0

print(f'image encoder : {t_encode*1000:7.1f} ms   (once per image)')
print(f'prompt + mask : {t_prompt*1000:7.1f} ms   (per click)')
print(f'speedup       : {t_encode/t_prompt:7.1f}x')

---

Every module here was written from the paper — patch embedding, decomposed relative position embeddings, windowed attention with padding, the two-way transformer, and hypernetwork-generated mask classifiers. See the repo README for the architecture walkthrough.